# WM-811K 최종 실험 v1

이 노트북은 WM-811K Wafer Map Dataset을 사용한다.

- lot 정의: lotName
- split: lotName 기준 group split
- target: failureType = none 정상, 그 외 failure pattern 불량
- 모델: 기존 저장 모델 XGBoost, LightGBM, CatBoost를 불러와 사용
- 비교: 0.5, F2최적화, OMMA, 제안

## 1. 환경 설정
필요한 경로와 라이브러리를 불러온다. 기존 저장 모델을 사용하므로 모델 학습은 수행하지 않는다.

In [ ]:
from pathlib import Path
import os
import sys
import pickle
import math

PROJECT_ROOT = Path.cwd()
DATA_ROOT = Path(os.environ.get("THESIS_DATA_ROOT", str(PROJECT_ROOT.parent)))
RUN_ROOT = PROJECT_ROOT / "experiments"
RESULT_ROOT = PROJECT_ROOT / "outputs" / "results"
MODEL_ROOT = PROJECT_ROOT / "outputs" / "models"

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(1, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from experiments.srs_ta_common import model_scores
from thesis_policy.omma_final_utils import omma_predict as omma_predict_online

RESULT_ROOT.mkdir(parents=True, exist_ok=True)


## 2. 평가 함수
SRS-TA, OMMA, 0.5, F2최적화 비교를 위한 공통 함수를 정의한다.

In [ ]:
THRESHOLD_GRID = np.arange(0.001, 1.0, 0.001)
MODELS = ["XGBoost", "LightGBM", "CatBoost"]


def classification_metrics(y_true, pred):
    y_true = np.asarray(y_true).astype(int)
    pred = np.asarray(pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    f2 = 5 * precision * recall / (4 * precision + recall) if precision + recall else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
        "gmean": math.sqrt(recall * specificity),
        "fpr": fp / (fp + tn) if fp + tn else 0.0,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }


def best_static_threshold(y_true, scores):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)
    pred = scores[None, :] >= THRESHOLD_GRID[:, None]
    tp = pred @ y_true
    fp = pred.sum(axis=1) - tp
    fn = y_true.sum() - tp
    denom = 5 * tp + 4 * fn + fp
    f2 = np.divide(5 * tp, denom, out=np.zeros_like(denom, dtype=float), where=denom != 0)
    best_idx = int(np.argmax(f2))
    return float(THRESHOLD_GRID[best_idx]), float(f2[best_idx])


def best_threshold_or_default(y_true, scores, default_theta=0.5):
    y_true = np.asarray(y_true).astype(int)
    if len(y_true) == 0 or y_true.sum() == 0:
        return float(default_theta)
    theta, _ = best_static_threshold(y_true, scores)
    return theta


def lot_reject_sequence(part, lot_col):
    return (
        part.groupby(lot_col, sort=False)
        .agg(lot_order=("order", "min"), reject=("target", "max"))
        .sort_values("lot_order")
    )


def states_by_lot(part, lot_col):
    lots = lot_reject_sequence(part, lot_col)
    state = "Normal"
    accept_count_tight = 0
    reject_history = []
    state_by_lot = {}
    for lot_id, row in lots.iterrows():
        state_by_lot[lot_id] = state
        reject = int(row["reject"])
        reject_history.append(reject)

        if state == "Tightened" and reject == 0:
            accept_count_tight += 1
        elif reject == 1 or state == "Normal":
            accept_count_tight = 0

        if len(reject_history) >= 5:
            recent = sum(reject_history[-5:])
            if state == "Normal" and recent >= 2:
                state = "Tightened"
                accept_count_tight = 0
            elif state == "Tightened" and accept_count_tight >= 5:
                state = "Normal"
                accept_count_tight = 0

    return part[lot_col].map(state_by_lot).to_numpy()


def srs_predict(scores, states, theta_tight):
    thresholds = np.where(np.asarray(states) == "Tightened", theta_tight, 0.5)
    return (np.asarray(scores) >= thresholds).astype(int)


def best_srs_threshold(y_true, scores, states):
    best_theta, best_f2 = 0.5, -1.0
    for theta in np.arange(0.001, 0.501, 0.001):
        pred = srs_predict(scores, states, theta)
        f2 = classification_metrics(y_true, pred)["f2"]
        if f2 > best_f2:
            best_theta, best_f2 = float(theta), float(f2)
    return best_theta, best_f2


def omma_predict_scores(y_true, scores, objective="f2"):
    return omma_predict_online(y_true, scores, objective=objective, limited=False)


def prepare_xy(df, feature_cols):
    parts = {
        split: df[df["split"].eq(split)].sort_values("order").reset_index(drop=True)
        for split in ["train", "val", "test"]
    }
    x_train_raw = parts["train"][feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x_train = imputer.fit_transform(x_train_raw)
    data = {
        "X_train": scaler.fit_transform(x_train),
        "y_train": parts["train"]["target"].to_numpy(dtype=int),
    }
    for split in ["val", "test"]:
        x_raw = parts[split][feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float64)
        data[f"X_{split}"] = scaler.transform(imputer.transform(x_raw))
        data[f"y_{split}"] = parts[split]["target"].to_numpy(dtype=int)
    audit = pd.DataFrame(
        [
            {
                "split": split,
                "rows": len(part),
                "positive": int(part["target"].sum()),
                "positive_rate": float(part["target"].mean()),
                "lots": int(part[LOT_COL].nunique()),
            }
            for split, part in parts.items()
        ]
    )
    return data, parts, audit


def load_saved_model(dataset_name, model_name):
    model_path = MODEL_ROOT / dataset_name / f"{model_name}.pkl"
    with model_path.open("rb") as f:
        return pickle.load(f)


def evaluate_dataset(dataset_name, df, feature_cols, lot_col):
    global LOT_COL
    LOT_COL = lot_col
    out_dir = RESULT_ROOT / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    data, parts, audit = prepare_xy(df, feature_cols)
    audit.to_csv(out_dir / "dataset_audit.csv", index=False, encoding="utf-8-sig")

    states = {split: states_by_lot(parts[split], lot_col) for split in ["train", "val", "test"]}
    pd.DataFrame(
        [
            {
                "split": split,
                "normal_rows": int((arr == "Normal").sum()),
                "tightened_rows": int((arr == "Tightened").sum()),
            }
            for split, arr in states.items()
        ]
    ).to_csv(out_dir / "state_counts.csv", index=False, encoding="utf-8-sig")

    rows = []
    thresholds = []
    for model_name in MODELS:
        model = load_saved_model(dataset_name, model_name)
        val_scores = model_scores(model, data["X_val"])
        theta_f2, val_f2 = best_static_threshold(data["y_val"], val_scores)
        theta_srs, val_srs_f2 = best_srs_threshold(data["y_val"], val_scores, states["val"])
        thresholds.append(
            {
                "model": model_name,
                "theta_f2": theta_f2,
                "theta_tight": theta_srs,
                "val_f2_static": val_f2,
                "val_f2_srs": val_srs_f2,
            }
        )

        for split in ["train", "val", "test"]:
            y = data[f"y_{split}"]
            scores = model_scores(model, data[f"X_{split}"])
            omma_pred, omma_thresholds = omma_predict_scores(y, scores, objective="f2")
            methods = {
                "0.5": (0.5, (scores >= 0.5).astype(int)),
                "F2최적화": (theta_f2, (scores >= theta_f2).astype(int)),
                "OMMA": (float(np.mean(omma_thresholds)), omma_pred),
                "제안": (theta_srs, srs_predict(scores, states[split], theta_srs)),
            }
            for method, (theta, pred) in methods.items():
                row = {
                    "dataset": dataset_name,
                    "model": model_name,
                    "split": split,
                    "method": method,
                    "threshold": theta,
                }
                row.update(classification_metrics(y, pred))
                rows.append(row)

    results = pd.DataFrame(rows)
    thresholds = pd.DataFrame(thresholds)
    results.to_csv(out_dir / "all_results.csv", index=False, encoding="utf-8-sig")
    thresholds.to_csv(out_dir / "thresholds.csv", index=False, encoding="utf-8-sig")
    test = results[results["split"].eq("test")].copy()
    test.to_csv(out_dir / "test_results.csv", index=False, encoding="utf-8-sig")
    return results, thresholds, audit


def test_table(results):
    cols = ["model", "method", "threshold", "precision", "recall", "f2", "gmean", "fpr", "tp", "fp", "fn", "tn"]
    return results[results["split"].eq("test")][cols].reset_index(drop=True)

## 3. 데이터 로딩 및 lot 구성
데이터셋별 lot 정의와 feature column을 구성한다.

In [ ]:
from run_wm_bd_indpensim import build_wm811k

DATASET_NAME, df = build_wm811k()
LOT_COL = "lotName"

exclude_cols = {"target", "split", "order", "lotName", "lot_order", "label_text", "split_strategy"}
feature_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
df = df.sort_values("order").reset_index(drop=True)

print("rows:", len(df))
print("features:", len(feature_cols))
print("lots:", df[LOT_COL].nunique())
df[["split", "target", LOT_COL, "waferIndex"]].head()

## 4. 기존 저장 모델 기반 평가
저장된 XGBoost, LightGBM, CatBoost 모델을 불러와 test 결과를 계산하고 result 폴더에 저장한다.

In [ ]:
results, thresholds, audit = evaluate_dataset(DATASET_NAME, df, feature_cols, LOT_COL)
print("saved result dir:", RESULT_ROOT / DATASET_NAME)
audit

## 5. Test 성능표
주요 지표는 Precision, Recall, F2, Gmean이며 FPR과 confusion matrix count도 함께 저장한다.

In [ ]:
table = test_table(results)
table

## 6. 비교 그래프
모델별 method 성능을 bar chart로 확인한다.

In [ ]:
plot_df = table.melt(
    id_vars=["model", "method"],
    value_vars=["precision", "recall", "f2", "gmean"],
    var_name="metric",
    value_name="score",
)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, metric in zip(axes.ravel(), ["precision", "recall", "f2", "gmean"]):
    sub = plot_df[plot_df["metric"].eq(metric)]
    sns.barplot(data=sub, x="model", y="score", hue="method", ax=ax)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right")
plt.show()

## 7. 논문용 Figure 생성 및 저장

아래 셀은 test 결과를 기준으로 논문에 넣을 수 있는 figure를 생성하고 저장한다.

저장 위치:

`최종모델/최종모델 실험/result/<데이터셋>/figures`

생성 figure:

- `performance_bars_test.png`: 모델별 성능 bar graph
- `pairwise_comparison_test.png`: 0.5, F2최적화, OMMA와 제안방법의 pairwise 비교
- `confusion_matrices_test.png`: 모델 및 방법별 confusion matrix
- `score_distribution_test.png`: test score 분포와 임계선
- `split_label_distribution.png`: train/validation/test 라벨 분포
- `state_distribution.png`: Normal/Tightened 검사상태 분포
- `lot_reject_sequence_test.png`: test lot 순서별 reject 이력과 검사상태


In [ ]:
FIG_DIR = RESULT_ROOT / DATASET_NAME / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ["0.5", "F2최적화", "OMMA", "제안"]
METRIC_ORDER = ["precision", "recall", "f2", "gmean"]
MODEL_ORDER = ["XGBoost", "LightGBM", "CatBoost"]

import matplotlib.font_manager as fm

font_path = Path(r"C:\Windows\Fonts\malgun.ttf")
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


def save_show(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("saved:", path)


test = test_table(results).copy()
test["method"] = pd.Categorical(test["method"], METHOD_ORDER, ordered=True)
test["model"] = pd.Categorical(test["model"], MODEL_ORDER, ordered=True)
test = test.sort_values(["model", "method"]).reset_index(drop=True)


# 1. 모델별 성능 bar graph
plot_df = test.melt(
    id_vars=["model", "method"],
    value_vars=METRIC_ORDER,
    var_name="metric",
    value_name="score",
)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, metric in zip(axes.ravel(), METRIC_ORDER):
    sub = plot_df[plot_df["metric"].eq(metric)]
    sns.barplot(data=sub, x="model", y="score", hue="method", hue_order=METHOD_ORDER, ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel("ML model")
    ax.set_ylabel("score")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)
save_show(fig, "performance_bars_test.png")


# 2. 비교군별 pairwise 성능 비교
pairs = [("0.5", "제안"), ("F2최적화", "제안"), ("OMMA", "제안")]
fig, axes = plt.subplots(len(pairs), len(METRIC_ORDER), figsize=(18, 10), constrained_layout=True)
for row_idx, (base_method, proposed_method) in enumerate(pairs):
    pair_df = test[test["method"].isin([base_method, proposed_method])].copy()
    for col_idx, metric in enumerate(METRIC_ORDER):
        ax = axes[row_idx, col_idx]
        sns.barplot(
            data=pair_df,
            x="model",
            y=metric,
            hue="method",
            hue_order=[base_method, proposed_method],
            ax=ax,
        )
        ax.set_title(f"{base_method} vs 제안 - {metric.upper()}")
        ax.set_xlabel("ML model")
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.25)
        if col_idx == len(METRIC_ORDER) - 1:
            ax.legend(loc="lower right", fontsize=8)
        else:
            ax.get_legend().remove()
save_show(fig, "pairwise_comparison_test.png")


# 3. Confusion matrix
fig, axes = plt.subplots(len(MODEL_ORDER), len(METHOD_ORDER), figsize=(15, 10), constrained_layout=True)
for r, model_name in enumerate(MODEL_ORDER):
    for c, method_name in enumerate(METHOD_ORDER):
        ax = axes[r, c]
        row = test[(test["model"].astype(str).eq(model_name)) & (test["method"].astype(str).eq(method_name))]
        if row.empty:
            ax.axis("off")
            continue
        row = row.iloc[0]
        cm = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]], dtype=int)
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"],
            ax=ax,
        )
        ax.set_title(f"{model_name} / {method_name}")
        ax.set_xlabel("")
        ax.set_ylabel("")
save_show(fig, "confusion_matrices_test.png")


# 4. Test score distribution and threshold lines
data, parts, audit_for_fig = prepare_xy(df, feature_cols)
states = {split: states_by_lot(parts[split], LOT_COL) for split in ["train", "val", "test"]}

fig, axes = plt.subplots(len(MODEL_ORDER), 1, figsize=(13, 11), constrained_layout=True)
for ax, model_name in zip(axes, MODEL_ORDER):
    model = load_saved_model(DATASET_NAME, model_name)
    scores = model_scores(model, data["X_test"])
    score_df = pd.DataFrame({"score": scores, "target": data["y_test"]})
    sns.histplot(
        data=score_df,
        x="score",
        hue="target",
        bins=50,
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.25,
        ax=ax,
    )
    th = thresholds[thresholds["model"].eq(model_name)].iloc[0]
    for value, label, color in [
        (0.5, "0.5", "black"),
        (float(th["theta_f2"]), "F2최적화", "#d95f02"),
        (float(th["theta_tight"]), "제안 θ_tight", "#1b9e77"),
    ]:
        ax.axvline(value, color=color, linestyle="--", linewidth=1.5, label=f"{label}: {value:.3f}")
    ax.set_title(f"{model_name} test score distribution")
    ax.set_xlabel("ML defect score")
    ax.set_ylabel("density")
    ax.set_xlim(-0.02, 1.02)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="upper center", ncol=4, fontsize=8)
save_show(fig, "score_distribution_test.png")


# 5. Split별 라벨 분포
audit_plot = audit.copy()
audit_plot["negative"] = audit_plot["rows"] - audit_plot["positive"]
count_df = audit_plot.melt(
    id_vars=["split"],
    value_vars=["negative", "positive"],
    var_name="label",
    value_name="count",
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
sns.barplot(data=count_df, x="split", y="count", hue="label", ax=axes[0])
axes[0].set_title("Label count by split")
axes[0].set_xlabel("split")
axes[0].set_ylabel("count")
axes[0].grid(axis="y", alpha=0.25)
sns.barplot(data=audit_plot, x="split", y="positive_rate", ax=axes[1], color="#4c78a8")
axes[1].set_title("Positive rate by split")
axes[1].set_xlabel("split")
axes[1].set_ylabel("positive rate")
axes[1].set_ylim(0, min(1.0, max(0.05, audit_plot["positive_rate"].max() * 1.25)))
axes[1].grid(axis="y", alpha=0.25)
save_show(fig, "split_label_distribution.png")


# 6. 검사 상태 분포
state_df = pd.read_csv(RESULT_ROOT / DATASET_NAME / "state_counts.csv")
state_long = state_df.melt(
    id_vars=["split"],
    value_vars=["normal_rows", "tightened_rows"],
    var_name="state",
    value_name="rows",
)
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
sns.barplot(data=state_long, x="split", y="rows", hue="state", ax=ax)
ax.set_title("Inspection state distribution")
ax.set_xlabel("split")
ax.set_ylabel("rows")
ax.grid(axis="y", alpha=0.25)
save_show(fig, "state_distribution.png")


# 7. Test lot reject sequence and state transition
test_part = parts["test"].copy()
test_part["state"] = states["test"]
lot_seq = (
    test_part.groupby(LOT_COL, sort=False)
    .agg(
        lot_order=("order", "min"),
        reject=("target", "max"),
        positive_rate=("target", "mean"),
        state=("state", "first"),
    )
    .sort_values("lot_order")
    .reset_index(drop=True)
)
lot_seq["lot_index"] = np.arange(1, len(lot_seq) + 1)
lot_seq["rolling_reject_rate"] = lot_seq["reject"].rolling(5, min_periods=1).mean()
fig, ax1 = plt.subplots(figsize=(14, 4.8), constrained_layout=True)
ax1.plot(lot_seq["lot_index"], lot_seq["rolling_reject_rate"], color="#1f77b4", linewidth=1.5, label="Rolling reject rate (5 lots)")
ax1.scatter(
    lot_seq.loc[lot_seq["reject"].eq(1), "lot_index"],
    lot_seq.loc[lot_seq["reject"].eq(1), "reject"],
    color="#d62728",
    s=14,
    label="Rejected lot",
)
ax1.set_xlabel("Test lot order")
ax1.set_ylabel("reject / rolling rate")
ax1.set_ylim(-0.05, 1.05)
ax1.grid(axis="y", alpha=0.25)
ax2 = ax1.twinx()
ax2.fill_between(
    lot_seq["lot_index"],
    0,
    (lot_seq["state"].eq("Tightened")).astype(int),
    color="#ffbb78",
    alpha=0.35,
    step="pre",
    label="Tightened state",
)
ax2.set_ylim(-0.05, 1.05)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(["Normal", "Tightened"])
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
ax1.set_title("Test lot reject sequence and inspection state")
save_show(fig, "lot_reject_sequence_test.png")